# Data Manipulation – Spain Energy Datasets
Continues from the cleaning notebook. It takes the 6 cleaned files (2015 onwards) and:
1. renames columns to short names
2. merges everything into **one table by Year**
3. removes columns that are repeated across files
4. creates new calculated columns
5. summarises, filters and saves the result

**Run the cleaning notebook first** so the `cleaned/` folder exists.

## 1. Setup

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

CLEAN_DIR = Path("cleaned")          # output folder of the cleaning notebook
OUT_DIR = Path("manipulated")
OUT_DIR.mkdir(exist_ok=True)

# cleaned file for each dataset (change the suffix if your names differ)
FILES = {
    "solar":       "1_solar_1_cleaned.csv",
    "wind":        "2_wind_1_cleaned.csv",
    "consumption": "3_electricity_consumption_1_cleaned.csv",
    "fuel":        "4_fuel_burned_1_cleaned.csv",
    "imports":     "5_imports_1_cleaned.csv",
    "gas":         "6_gas_1_cleaned.csv",
}

## 2. Load the cleaned files

In [ ]:
data = {k: pd.read_csv(CLEAN_DIR / f) for k, f in FILES.items()}

for k, df in data.items():
    print(f"{k:12s} shape={df.shape}  years={df['Year'].min()}-{df['Year'].max()}")

## 3. Rename columns
Short `snake_case` names are easier to type and use in code. Units are kept in the name.

In [ ]:
RENAME = {
    # solar
    "Solar electricity generation (TWh)": "solar_gen_twh",
    "Solar share of electricity (%)": "solar_share_pct",
    "Solar primary energy (TWh, substitution method)": "solar_primary_twh",
    "Solar electricity per capita (kWh)": "solar_per_capita_kwh",
    # wind
    "Wind electricity generation (TWh)": "wind_gen_twh",
    "Wind share of electricity (%)": "wind_share_pct",
    "Wind primary energy (TWh, substitution method)": "wind_primary_twh",
    "Wind electricity per capita (kWh)": "wind_per_capita_kwh",
    # consumption
    "Electricity demand (TWh)": "demand_twh",
    "Electricity generation (TWh)": "generation_twh",
    "Electricity generation per capita (kWh)": "generation_per_capita_kwh",
    "Electricity demand per capita (kWh)": "demand_per_capita_kwh",
    "Population": "population",
    "GDP (int-$, 2011 prices)": "gdp",
    # fuel burned
    "Electricity from fossil fuels (TWh)": "fossil_elec_twh",
    "Electricity from gas (TWh)": "gas_elec_twh",
    "Electricity from oil (TWh)": "oil_elec_twh",
    "Electricity from coal (TWh)": "coal_elec_twh",
    "Fossil share of electricity (%)": "fossil_share_pct",
    "Fossil fuel primary energy consumption (TWh)": "fossil_primary_twh",
    "Gas consumption (TWh)": "gas_consumption_twh",
    "Oil consumption (TWh)": "oil_consumption_twh",
    "Coal consumption (TWh)": "coal_consumption_twh",
    # imports
    "Net electricity imports (TWh)": "net_elec_imports_twh",
    "Net electricity imports share of demand (%)": "net_elec_imports_share_pct",
    "Gas production (TWh)": "gas_production_twh",
    "Oil production (TWh)": "oil_production_twh",
    "DERIVED gas implied net imports = consumption - production (TWh)": "gas_net_imports_twh",
    "DERIVED oil implied net imports = consumption - production (TWh)": "oil_net_imports_twh",
    # gas
    "Gas share of electricity (%)": "gas_share_elec_pct",
    "Gas share of primary energy (%)": "gas_share_primary_pct",
    "Gas energy per capita (kWh)": "gas_per_capita_kwh",
}

data = {k: df.rename(columns=RENAME) for k, df in data.items()}

# make sure every column was renamed
leftover = [c for df in data.values() for c in df.columns if c != "Year" and c not in RENAME.values()]
print("columns not renamed:", leftover)

## 4. Merge into one table
Some columns appear in more than one file (e.g. gas consumption is in files 4, 5 and 6). First we **check they are identical**, then keep only one copy.

In [ ]:
# find columns that exist in more than one file
seen = {}
for k, df in data.items():
    for c in df.columns:
        if c != "Year":
            seen.setdefault(c, []).append(k)
repeated = {c: ks for c, ks in seen.items() if len(ks) > 1}
print("repeated columns:", repeated)

# verify the copies really hold the same values
for c, ks in repeated.items():
    base = data[ks[0]].set_index("Year")[c]
    for k in ks[1:]:
        other = data[k].set_index("Year")[c]
        assert base.equals(other), f"{c} differs between {ks[0]} and {k}"
print("OK: all repeated columns are identical")

In [ ]:
# merge on Year, dropping the repeated columns from later files
merged = data["solar"]
for k in ["wind", "consumption", "fuel", "imports", "gas"]:
    df = data[k]
    df = df.drop(columns=[c for c in df.columns if c != "Year" and c in merged.columns])
    merged = merged.merge(df, on="Year", how="outer", validate="one_to_one")

merged = merged.sort_values("Year").reset_index(drop=True)
print("merged shape:", merged.shape)
merged.head()

## 5. Create new columns

In [ ]:
df = merged.copy()

# --- renewables (solar + wind) ---
df["renewable_gen_twh"] = df["solar_gen_twh"] + df["wind_gen_twh"]
df["renewable_share_pct"] = df["renewable_gen_twh"] / df["generation_twh"] * 100

# --- electricity balance ---
df["self_sufficiency_pct"] = df["generation_twh"] / df["demand_twh"] * 100   # >100 = net exporter
df["net_position"] = df["net_elec_imports_twh"].apply(
    lambda x: "importer" if x > 0 else ("exporter" if x < 0 else "balanced")
)

# --- fuels ---
df["gas_import_dependency_pct"] = df["gas_net_imports_twh"] / df["gas_consumption_twh"] * 100
df["oil_import_dependency_pct"] = df["oil_net_imports_twh"] / df["oil_consumption_twh"] * 100
df["fossil_electricity_check_twh"] = df[["gas_elec_twh", "oil_elec_twh", "coal_elec_twh"]].sum(axis=1)

# --- economy ---
df["gdp_per_capita"] = df["gdp"] / df["population"]
df["electricity_intensity_kwh_per_gdp"] = df["demand_twh"] * 1e9 / df["gdp"]   # kWh per $ of GDP

# --- year-over-year growth (%) for key indicators ---
for col in ["demand_twh", "generation_twh", "solar_gen_twh", "wind_gen_twh", "renewable_gen_twh"]:
    df[f"{col}_yoy_pct"] = df[col].pct_change(fill_method=None) * 100

# --- period label ---
df["period"] = pd.cut(df["Year"], bins=[2014, 2019, 2025], labels=["2015-2019", "2020-2025"])

df.head()

## 6. Missing values
We do **not** invent numbers. Instead we list what is missing so you can decide how to treat it.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing.to_string())

# which years are affected?
for c in missing.index[:6]:
    print(f"{c:40s} missing in:", df.loc[df[c].isna(), "Year"].tolist())

Two optional ways to deal with the gaps (the main table above stays untouched):

In [ ]:
# Option A: version without the columns that are mostly empty (>= 50% missing)
keep = df.columns[df.isna().mean() < 0.5]
df_dense = df[keep]
print("columns dropped:", sorted(set(df.columns) - set(keep)))

# Option B: only the years that have no missing values at all
df_complete_years = df.dropna()
print("years with complete data:", df_complete_years["Year"].tolist())

## 7. Summaries and filtering

In [ ]:
# average by period
cols = ["demand_twh", "generation_twh", "renewable_share_pct", "self_sufficiency_pct", "gas_share_elec_pct"]
df.groupby("period", observed=True)[cols].mean().round(2)

In [ ]:
# years when Spain was a net exporter of electricity
df.loc[df["net_position"] == "exporter", ["Year", "net_elec_imports_twh", "self_sufficiency_pct"]]

In [ ]:
# best / worst year for a few indicators
for col in ["renewable_share_pct", "demand_twh", "gas_share_elec_pct"]:
    hi = df.loc[df[col].idxmax()]
    lo = df.loc[df[col].idxmin()]
    print(f"{col:22s} max: {hi['Year']} ({hi[col]:.2f})   min: {lo['Year']} ({lo[col]:.2f})")

## 8. Quick charts

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(df["Year"], df["solar_gen_twh"], marker="o", label="Solar")
ax[0].plot(df["Year"], df["wind_gen_twh"], marker="o", label="Wind")
ax[0].set_title("Solar and wind generation (TWh)")
ax[0].legend()

ax[1].plot(df["Year"], df["demand_twh"], marker="o", label="Demand")
ax[1].plot(df["Year"], df["generation_twh"], marker="o", label="Generation")
ax[1].set_title("Electricity demand vs generation (TWh)")
ax[1].legend()

for a in ax:
    a.set_xlabel("Year")
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Save

In [ ]:
df.to_csv(OUT_DIR / "spain_energy_merged.csv", index=False)
df_dense.to_csv(OUT_DIR / "spain_energy_merged_dense.csv", index=False)
print("saved ->", OUT_DIR / "spain_energy_merged.csv", df.shape)
print("saved ->", OUT_DIR / "spain_energy_merged_dense.csv", df_dense.shape)